# Retailrocket E-Commerce — SQLite Database Project
### Practical Jupyter Notebook | SQL design, real data ingestion, compound keys, randomized scenarios, NoSQL export and association rules

**Source:** The user-provided `Railrocket Recommender System.zip`, derived from the public [Retailrocket Recommender System dataset](https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset). This notebook reads all four CSVs from the ZIP **without extracting the ~1 GB of CSV files**.

**Research question:** How do ecommerce visitors browse, add products to their carts, and buy; which product categories co-occur in inferred browsing sessions?

**Important:** Real-world source events are distinguished from **explicitly synthetic** marketing promotions. Item identifiers are anonymized. This dataset does **not** contain original product prices, demographic profiles, or actual session IDs. Sessions are inferred using a 30-minute inactivity gap, and association rules based on those sessions represent **co-interactions**, not confirmed co-purchases. The training sample selects visitors consistently, keeping all of each selected visitor's observed events.

## Assessment checklist

- **SQLite DB:** `retailrocket_project.sqlite`, with 8 relational tables and a >1,000-row `events` table.
- **Nominal:** `events.event_type` (view / addtocart / transaction).
- **Ordinal:** `events.intent_level` (Low < Medium < High), a **derived analytical assumption** about action intent.
- **Interval:** `events.event_utc` (calendar date/time; absolute zero is arbitrary) and `calendar_year`.
- **Ratio:** `events.prior_session_events`, `sessions.event_count`, `visitors.total_events` (true zero; meaningful ratios).
- **Keys:** event PKs; visitor/item/session/category FKs; `item_property_history` and `visitor_promotions` compound PKs.
- **Randomized data:** Reproducible, separately labeled simulated marketing promotions and visitor assignments.
- **Retrieval:** `JOIN`, `GROUP BY`, `WHERE`, CTE and window-function queries.
- **NoSQL:** JSON-document export plus an **optional real MongoDB** ingestion/retrieval example.
- **Onward processing:** Data cleaning, CSV exports, descriptive statistics, and session-based association rules.

## 0 · Setup and reproducibility
Put this notebook and `Railrocket Recommender System.zip` in the same directory, or keep the ZIP at the given `/mnt/data/` path. Core cells need Python 3.10+, pandas, NumPy and Matplotlib. `sqlite3`, `zipfile` and `itertools` are from the Python standard library. The optional MongoDB demonstration requires `pymongo` and a running MongoDB instance.

In [ ]:
import csv, io, json, sqlite3, zipfile
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 2026
rng = np.random.default_rng(SEED)
SAMPLE_RATE = 0.08  # hash-selected visitors; preserves complete histories for selected visitors
SESSION_GAP_MINUTES = 30
READ_CHUNK_SIZE = 250_000
SOURCE_NAMES = ["category_tree.csv", "events.csv", "item_properties_part1.csv", "item_properties_part2.csv"]

candidates = [Path("Railrocket Recommender System.zip"),
              Path("/mnt/data/Railrocket Recommender System.zip")]
ZIP_PATH = next((p for p in candidates if p.is_file()), None)
if ZIP_PATH is None:
    raise FileNotFoundError("Put 'Railrocket Recommender System.zip' next to the notebook and rerun.")
OUTPUT_DIR = Path.cwd() / "retailrocket_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
DB_PATH = OUTPUT_DIR / "retailrocket_project.sqlite"

with zipfile.ZipFile(ZIP_PATH) as archive:
    all_names = archive.namelist()
    name_map = {short: next((n for n in all_names if Path(n).name.lower() == short.lower()), None)
                for short in SOURCE_NAMES}
    if not name_map["events.csv"]:
        raise FileNotFoundError("events.csv was not found inside the uploaded ZIP")
    display(pd.DataFrame([{"source_file": n, "zip_member": name_map[n],
                           "size_mb": round(archive.getinfo(name_map[n]).file_size / 1_000_000, 2)
                           if name_map[n] else None} for n in SOURCE_NAMES]))
print("Output directory:", OUTPUT_DIR)
print("Input ZIP:", ZIP_PATH)

## 1 · Read and prepare a real, representative sample of visitor activity
We select visitors by a deterministic hash, not by taking random independent events. This is important: independent event sampling would artificially break individual browsing histories and association baskets. Source IDs are preserved.

In [ ]:
parts = []
raw_total = 0
with zipfile.ZipFile(ZIP_PATH) as archive, archive.open(name_map["events.csv"]) as stream:
    chunks = pd.read_csv(stream, chunksize=READ_CHUNK_SIZE,
                         dtype={"timestamp": "int64", "visitorid": "int64",
                                "event": "string", "itemid": "int64", "transactionid": "Int64"})
    for chunk in chunks:
        raw_total += len(chunk)
        visitor_hash = (chunk["visitorid"] * 2654435761) % (2**32)
        keep_visitor = visitor_hash < (SAMPLE_RATE * (2**32))
        sample = chunk.loc[keep_visitor].copy()
        # Source record position is unique, even when event values are identical.
        sample["source_row_number"] = sample.index.to_numpy() + 2
        parts.append(sample)
raw = pd.concat(parts, ignore_index=True)
assert len(raw) >= 1000, "Too few records: increase SAMPLE_RATE."
print(f"Original events: {raw_total:,}; sampled events: {len(raw):,}; "
      f"unique sampled visitors: {raw['visitorid'].nunique():,}")
display(raw.head())

In [ ]:
# Data-quality checks and transformations — preserve legitimate NULL transaction IDs.
quality_before = pd.DataFrame({
    "missing": raw.isna().sum(),
    "dtype": raw.dtypes.astype(str)
})
display(quality_before)

valid_types = {"view", "addtocart", "transaction"}
events = raw.loc[
    raw["event"].isin(valid_types) & raw["timestamp"].gt(0) &
    raw["visitorid"].ge(0) & raw["itemid"].ge(0)
].copy()
events = events.drop_duplicates(subset=["source_row_number"])
events["event_utc"] = pd.to_datetime(events["timestamp"], unit="ms", utc=True,
                                     errors="coerce")
events = events.dropna(subset=["event_utc"])
events["calendar_year"] = events["event_utc"].dt.year.astype(int)
events["event_utc_text"] = events["event_utc"].dt.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
intent_map = {"view": ("Low", 1), "addtocart": ("Medium", 2),
              "transaction": ("High", 3)}
events["intent_level"] = events["event"].map(lambda x: intent_map[x][0])
events["intent_rank"] = events["event"].map(lambda x: intent_map[x][1]).astype(int)

# Infer sessions from elapsed time between activities of the SAME visitor.
events = events.sort_values(["visitorid", "timestamp", "source_row_number"]).reset_index(drop=True)
previous_timestamp = events.groupby("visitorid")["timestamp"].shift(1)
events["new_session"] = (previous_timestamp.isna() |
                         ((events["timestamp"] - previous_timestamp) > SESSION_GAP_MINUTES*60_000))
events["session_number"] = events.groupby("visitorid")["new_session"].cumsum().astype(int)
events["session_id"] = events["visitorid"].astype(str) + ":" + events["session_number"].astype(str)
events["prior_session_events"] = events.groupby("session_id").cumcount().astype(int)
print("Retained valid events:", len(events), "| inferred sessions:", events.session_id.nunique())
display(events[["visitorid","event","event_utc_text","session_id", "prior_session_events"]].head(10))

## 2 · Relational design (schema)

**Relationships:** `visitors` 1—many `sessions`; `sessions` 1—many `events`; `items` 1—many `events`; `categories` 1—many `items`; `items` 1—many `item_property_history`; `visitors` many—many `promotions` through `visitor_promotions`.

**Why a compound key?** One item can have multiple properties at multiple times. `(item_id, property_name, effective_ts_ms)` uniquely identifies a historical item-property observation. The `(visitor_id, promotion_id)` key prevents duplicate simulated campaign assignments. SQLite foreign-key checking is deliberately turned on before creating tables. Tables are rebuilt on every notebook run for reproducible results.

In [ ]:
if DB_PATH.exists():
    DB_PATH.unlink()  # Reproducible rebuild; removes ONLY the database produced by this notebook.
con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA foreign_keys = ON")
assert con.execute("PRAGMA foreign_keys").fetchone()[0] == 1
con.executescript("""
CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY,
    parent_id INTEGER,
    FOREIGN KEY (parent_id) REFERENCES categories(category_id) DEFERRABLE INITIALLY DEFERRED
);
CREATE TABLE visitors (
    visitor_id INTEGER PRIMARY KEY,
    first_seen_utc TEXT NOT NULL,
    last_seen_utc TEXT NOT NULL,
    total_events INTEGER NOT NULL CHECK (total_events >= 0),
    unique_items INTEGER NOT NULL CHECK (unique_items >= 0),
    transactions INTEGER NOT NULL CHECK (transactions >= 0),
    activity_tier TEXT NOT NULL CHECK (activity_tier IN ('Low','Medium','High'))
);
CREATE TABLE items (
    item_id INTEGER PRIMARY KEY,
    category_id INTEGER,
    available_last_seen INTEGER CHECK (available_last_seen IN (0,1)),
    observed_events INTEGER NOT NULL CHECK (observed_events >= 0),
    popularity_tier TEXT NOT NULL CHECK (popularity_tier IN ('Low','Medium','High')),
    FOREIGN KEY(category_id) REFERENCES categories(category_id)
);
CREATE TABLE sessions (
    session_id TEXT PRIMARY KEY,
    visitor_id INTEGER NOT NULL,
    started_utc TEXT NOT NULL,
    ended_utc TEXT NOT NULL,
    event_count INTEGER NOT NULL CHECK (event_count >= 1),
    duration_seconds REAL NOT NULL CHECK (duration_seconds >= 0),
    FOREIGN KEY(visitor_id) REFERENCES visitors(visitor_id)
);
CREATE TABLE events (
    event_id INTEGER PRIMARY KEY,
    visitor_id INTEGER NOT NULL,
    item_id INTEGER NOT NULL,
    session_id TEXT NOT NULL,
    event_type TEXT NOT NULL CHECK (event_type IN ('view','addtocart','transaction')),
    intent_level TEXT NOT NULL CHECK (intent_level IN ('Low','Medium','High')),
    intent_rank INTEGER NOT NULL CHECK (intent_rank BETWEEN 1 AND 3),
    event_ts_ms INTEGER NOT NULL CHECK (event_ts_ms > 0),
    event_utc TEXT NOT NULL,
    calendar_year INTEGER NOT NULL,
    prior_session_events INTEGER NOT NULL CHECK (prior_session_events >= 0),
    transaction_id INTEGER,
    FOREIGN KEY(visitor_id) REFERENCES visitors(visitor_id),
    FOREIGN KEY(item_id) REFERENCES items(item_id),
    FOREIGN KEY(session_id) REFERENCES sessions(session_id)
);
CREATE TABLE item_property_history (
    item_id INTEGER NOT NULL,
    property_name TEXT NOT NULL,
    effective_ts_ms INTEGER NOT NULL,
    property_value TEXT NOT NULL,
    PRIMARY KEY(item_id, property_name, effective_ts_ms),
    FOREIGN KEY(item_id) REFERENCES items(item_id)
);
CREATE TABLE promotions (
    promotion_id INTEGER PRIMARY KEY,
    promotion_name TEXT NOT NULL,
    priority_level TEXT NOT NULL CHECK (priority_level IN ('Standard','Featured','Premium')),
    simulated_discount_pct REAL NOT NULL CHECK (simulated_discount_pct BETWEEN 0 AND 100),
    simulated_budget_usd REAL NOT NULL CHECK (simulated_budget_usd >= 0),
    simulated INTEGER NOT NULL DEFAULT 1 CHECK (simulated = 1)
);
CREATE TABLE visitor_promotions (
    visitor_id INTEGER NOT NULL,
    promotion_id INTEGER NOT NULL,
    assignment_utc TEXT NOT NULL,
    simulated_response_probability REAL NOT NULL CHECK (simulated_response_probability BETWEEN 0 AND 1),
    PRIMARY KEY(visitor_id, promotion_id),
    FOREIGN KEY(visitor_id) REFERENCES visitors(visitor_id),
    FOREIGN KEY(promotion_id) REFERENCES promotions(promotion_id)
);
CREATE INDEX ix_events_visitor_time ON events(visitor_id, event_ts_ms);
CREATE INDEX ix_events_session ON events(session_id);
CREATE INDEX ix_events_item_type ON events(item_id, event_type);
CREATE INDEX ix_items_category ON items(category_id);
CREATE INDEX ix_properties_lookup ON item_property_history(item_id,property_name,effective_ts_ms);
""")
con.commit()
print("Created database:", DB_PATH)

## 3 · Populate reference tables and load event data
Visitors, sessions and item-popularity tiers are calculated **from real observed events** in the selected cohort. No original demographics, item prices or session IDs are invented.

In [ ]:
with zipfile.ZipFile(ZIP_PATH) as archive, archive.open(name_map["category_tree.csv"]) as stream:
    category_data = pd.read_csv(stream)
category_ids = set(category_data["categoryid"].dropna().astype(int))
category_rows = [(int(r.categoryid), None if pd.isna(r.parentid) else int(r.parentid))
                 for r in category_data.itertuples(index=False)]
con.executemany("INSERT INTO categories(category_id,parent_id) VALUES(?,?)", category_rows)

visitors = events.groupby("visitorid", sort=False).agg(
    first_seen_utc=("event_utc_text", "min"), last_seen_utc=("event_utc_text", "max"),
    total_events=("event", "size"), unique_items=("itemid", "nunique"),
    transactions=("event", lambda s: int((s == "transaction").sum()))
).reset_index()
visitors["activity_tier"] = np.select(
    [visitors.total_events <= 1, visitors.total_events <= 4], ["Low", "Medium"], default="High")
con.executemany("""INSERT INTO visitors
    (visitor_id,first_seen_utc,last_seen_utc,total_events,unique_items,transactions,activity_tier)
    VALUES(?,?,?,?,?,?,?)""", [tuple(row) for row in visitors.itertuples(index=False,name=None)])

items = events.groupby("itemid", sort=False).size().rename("observed_events").reset_index()
items["popularity_tier"] = np.select(
    [items.observed_events <= 1, items.observed_events <= 5], ["Low", "Medium"], default="High")
con.executemany("""INSERT INTO items(item_id,category_id,available_last_seen,observed_events,popularity_tier)
    VALUES(?,NULL,NULL,?,?)""", [tuple(row) for row in items[["itemid","observed_events","popularity_tier"]].itertuples(index=False,name=None)])

sessions = events.groupby("session_id", sort=False).agg(
    visitor_id=("visitorid", "first"), started_utc=("event_utc_text", "min"),
    ended_utc=("event_utc_text", "max"), event_count=("event", "size"),
    min_ts=("timestamp", "min"), max_ts=("timestamp", "max")
).reset_index()
sessions["duration_seconds"] = (sessions.max_ts - sessions.min_ts) / 1000.0
con.executemany("""INSERT INTO sessions(session_id,visitor_id,started_utc,ended_utc,event_count,duration_seconds)
    VALUES(?,?,?,?,?,?)""", [tuple(row) for row in sessions[["session_id","visitor_id","started_utc","ended_utc","event_count","duration_seconds"]].itertuples(index=False,name=None)])

selected_cols = ["source_row_number","visitorid","itemid","session_id","event","intent_level",
                 "intent_rank","timestamp","event_utc_text","calendar_year","prior_session_events","transactionid"]
clean_export = events[selected_cols].copy()
clean_export.columns = ["event_id","visitor_id","item_id","session_id","event_type","intent_level",
                        "intent_rank","event_ts_ms","event_utc","calendar_year","prior_session_events","transaction_id"]
# pandas extension Int64 requires conversion to Python None before sqlite3 parameter binding.
clean_export = clean_export.astype(object).where(pd.notna(clean_export), None)
con.executemany("""INSERT INTO events
 (event_id,visitor_id,item_id,session_id,event_type,intent_level,intent_rank,event_ts_ms,
  event_utc,calendar_year,prior_session_events,transaction_id)
 VALUES(?,?,?,?,?,?,?,?,?,?,?,?)""", clean_export.itertuples(index=False,name=None))
con.commit()
print("Inserted real source-based rows:", len(visitors), "visitors,", len(items),
      "items,", len(sessions), "sessions and", len(events), "events")

## 4 · Historical item properties (category / availability)
Original item properties are encoded and timestamped. We only import genuine `categoryid` and `available` values for items appearing in this cohort. The **latest recorded property value** is used for the current item snapshot. Earlier changes remain in a table with a compound primary key. Do **not** treat latest category or availability as necessarily valid at every historical event time.

In [ ]:
selected_item_ids = set(int(x) for x in items.itemid)
latest_property = {}  # (item_id, property_name) -> (effective_ts_ms, string_value)
property_rows_written = 0
for file_short in ("item_properties_part1.csv", "item_properties_part2.csv"):
    if name_map[file_short] is None:
        print("Missing optional property source:", file_short)
        continue
    with zipfile.ZipFile(ZIP_PATH) as archive, archive.open(name_map[file_short]) as stream:
        parts_iterator = pd.read_csv(stream, chunksize=READ_CHUNK_SIZE,
                                     dtype={"timestamp": "int64", "itemid": "int64",
                                            "property": "string", "value": "string"})
        for prop_chunk in parts_iterator:
            selected = prop_chunk.loc[
                prop_chunk["property"].isin(["categoryid", "available"]) &
                prop_chunk["itemid"].isin(selected_item_ids),
                ["itemid", "property", "timestamp", "value"]
            ].dropna().drop_duplicates(subset=["itemid","property","timestamp"], keep="last")
            if selected.empty:
                continue
            rows = [(int(i), str(prop), int(ts), str(val)) for i,prop,ts,val
                    in selected.itertuples(index=False,name=None)]
            con.executemany("""INSERT OR REPLACE INTO item_property_history
                (item_id,property_name,effective_ts_ms,property_value) VALUES (?,?,?,?)""", rows)
            property_rows_written += len(rows)
            for item_id, prop, ts, value in rows:
                key = (item_id, prop)
                if key not in latest_property or ts >= latest_property[key][0]:
                    latest_property[key] = (ts, value)
    con.commit()
    print("Finished scanning:", file_short)

category_updates = []
availability_updates = []
for (item_id, prop), (_ts, val) in latest_property.items():
    if prop == "categoryid":
        try:
            cat = int(val)
            if cat in category_ids: category_updates.append((cat, item_id))
        except ValueError:
            pass
    elif prop == "available":
        if val in {"0","1"}: availability_updates.append((int(val), item_id))
con.executemany("UPDATE items SET category_id=? WHERE item_id=?", category_updates)
con.executemany("UPDATE items SET available_last_seen=? WHERE item_id=?", availability_updates)
con.commit()
print(f"Imported property history rows: {con.execute('SELECT COUNT(*) FROM item_property_history').fetchone()[0]:,}")
print(f"Items with known latest category: {len(category_updates):,}; availability: {len(availability_updates):,}")

## 5 · Reproducible randomized *synthetic* marketing data
Randomized extensions make the model more realistic for exercises without altering any factual Retailrocket behaviour. All generated marketing tables are explicitly marked `simulated`, and they must not be cited as historical transactions or campaigns.

In [ ]:
campaign_names = ["Welcome offer", "Re-engagement", "Basket reminder",
                  "Loyal visitor", "Seasonal promotion", "Weekend discovery"]
levels = ["Standard", "Featured", "Premium", "Standard", "Featured", "Premium"]
promos = [(i+1, name, levels[i], round(float(rng.uniform(5,30)),2),
           round(float(rng.uniform(5_000,55_000)),2), 1)
          for i,name in enumerate(campaign_names)]
con.executemany("""INSERT INTO promotions(promotion_id,promotion_name,priority_level,
   simulated_discount_pct,simulated_budget_usd,simulated) VALUES(?,?,?,?,?,?)""", promos)

# Use a small randomized sample of actual visitors; each can have 1 or 2 assignments.
sample_size = min(4000, len(visitors))
selected_visitors = rng.choice(visitors.visitorid.to_numpy(), size=sample_size, replace=False)
assignments = []
for visitor_id in selected_visitors:
    n_campaigns = int(rng.choice([1, 2], p=[0.8, 0.2]))
    campaigns = rng.choice(np.arange(1, 7), size=n_campaigns, replace=False)
    for campaign_id in campaigns:
        assignments.append((int(visitor_id), int(campaign_id), "2015-09-01T00:00:00Z",
                            round(float(rng.beta(2, 7)), 4)))
con.executemany("""INSERT INTO visitor_promotions
(visitor_id,promotion_id,assignment_utc,simulated_response_probability)
VALUES (?,?,?,?)""", assignments)
con.commit()
display(pd.read_sql_query("SELECT * FROM promotions", con))
print("Simulated, not observed, visitor-campaign assignments:", len(assignments))

## 6 · Validate the assignment requirements and database integrity
The assertions fail visibly if the minimum row count, number of columns, measurement-scale constraints, compound keys, or foreign-key integrity are not satisfied.

In [ ]:
schema = pd.read_sql_query("""SELECT name, sql FROM sqlite_master
    WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name""", con)
table_summary = pd.read_sql_query("""
SELECT name AS table_name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%'
ORDER BY name""", con)
table_summary["row_count"] = table_summary.table_name.map(
    lambda table: con.execute(f'SELECT COUNT(*) FROM "{table}"').fetchone()[0])
table_summary["column_count"] = table_summary.table_name.map(
    lambda table: len(con.execute(f'PRAGMA table_info("{table}")').fetchall()))
display(table_summary)

assert con.execute("SELECT COUNT(*) FROM events").fetchone()[0] >= 1000
assert len(con.execute("PRAGMA table_info(events)").fetchall()) >= 7
assert con.execute("PRAGMA foreign_keys").fetchone()[0] == 1
assert con.execute("PRAGMA foreign_key_check").fetchall() == [], "FK violations detected"
assert con.execute("SELECT COUNT(*) FROM events WHERE prior_session_events < 0").fetchone()[0] == 0
assert con.execute("SELECT COUNT(*) FROM events WHERE event_type NOT IN ('view','addtocart','transaction')").fetchone()[0] == 0
assert con.execute("SELECT COUNT(*) FROM item_property_history").fetchone()[0] > 0
assert con.execute("SELECT COUNT(*) FROM visitor_promotions").fetchone()[0] > 0
print("ALL CHECKS PASSED: 1000+ rows, 7+ columns, constraints, real/derived measurement scales, compound PKs, FKs.")

print("\nDetailed event schema:")
display(pd.DataFrame(con.execute("PRAGMA table_info(events)").fetchall(),
                     columns=["cid","column","sql_type","not_null","default","primary_key"]))
print("Foreign-key relationships on events:")
display(pd.DataFrame(con.execute("PRAGMA foreign_key_list(events)").fetchall(),
                     columns=["id","seq","table","from","to","on_update","on_delete","match"]))

## 7 · SQL retrieval, joins, aggregations and window functions
These queries illustrate retrieving information directly from the generated SQLite database and provide ready-to-comment examples for the project report.

In [ ]:
queries = {
    "1. Nominal / ordinal event frequencies": """
        SELECT event_type, intent_level, intent_rank, COUNT(*) AS number_of_events
        FROM events GROUP BY event_type,intent_level,intent_rank ORDER BY intent_rank;""",
    "2. Join five tables: categories with top interactions": """
        SELECT c.category_id, COUNT(*) AS interactions,
               COUNT(DISTINCT e.visitor_id) AS unique_visitors,
               SUM(CASE WHEN e.event_type = 'transaction' THEN 1 ELSE 0 END) AS purchase_events
        FROM events e JOIN items i ON e.item_id=i.item_id
        JOIN categories c ON i.category_id=c.category_id
        JOIN sessions s ON s.session_id=e.session_id
        JOIN visitors v ON v.visitor_id=s.visitor_id
        GROUP BY c.category_id ORDER BY interactions DESC LIMIT 10;""",
    "3. Purchase funnel by event": """
        SELECT event_type, COUNT(*) AS events, COUNT(DISTINCT visitor_id) AS visitors
        FROM events GROUP BY event_type ORDER BY events DESC;""",
    "4. Top 10 returning visitors": """
        SELECT visitor_id, total_events, unique_items, transactions, activity_tier
        FROM visitors WHERE total_events > 1 ORDER BY total_events DESC LIMIT 10;""",
    "5. Monthly totals using SQLite date functions": """
        SELECT substr(event_utc,1,7) AS month_utc, COUNT(*) AS events,
               SUM(CASE WHEN event_type='transaction' THEN 1 ELSE 0 END) AS purchases
        FROM events GROUP BY substr(event_utc,1,7) ORDER BY month_utc;""",
    "6. Compound-key lookup and time-dependent properties": """
        SELECT item_id, property_name, effective_ts_ms, property_value
        FROM item_property_history WHERE property_name='categoryid'
        ORDER BY effective_ts_ms DESC LIMIT 10;""",
    "7. Window functions: visitor chronology": """
        SELECT visitor_id,event_type,event_utc,
               ROW_NUMBER() OVER (PARTITION BY visitor_id ORDER BY event_ts_ms,event_id) AS visitor_event_number
        FROM events WHERE visitor_id=(SELECT visitor_id FROM visitors ORDER BY total_events DESC LIMIT 1)
        ORDER BY event_ts_ms LIMIT 10;"""
}
for label, sql in queries.items():
    print("\n", label)
    display(pd.read_sql_query(sql, con))

## 8 · Data preparation and export for onward processing
Source missing transaction IDs correspond to non-purchase actions and are **not** automatically data-quality errors. The following export is a denormalized, query-derived analytical view. In contrast, the SQLite database remains relational and normalized.

In [ ]:
analysis_df = pd.read_sql_query("""
SELECT e.event_id,e.event_utc,e.calendar_year,e.visitor_id,v.activity_tier,
       e.item_id,i.category_id,i.available_last_seen,
       e.session_id,s.event_count AS session_event_count,
       e.event_type,e.intent_level,e.intent_rank,e.prior_session_events,
       e.transaction_id
FROM events e JOIN visitors v ON v.visitor_id=e.visitor_id
JOIN items i ON e.item_id=i.item_id
JOIN sessions s ON e.session_id=s.session_id
""", con)
analysis_df["event_utc"] = pd.to_datetime(analysis_df["event_utc"], utc=True)
analysis_df["is_purchase"] = analysis_df["event_type"].eq("transaction").astype(int)
analysis_df["has_known_category"] = analysis_df["category_id"].notna().astype(int)
clean_csv = OUTPUT_DIR / "events_analysis_ready.csv"
analysis_df.to_csv(clean_csv, index=False)
print("Exported:", clean_csv, "shape:", analysis_df.shape)
display(analysis_df.head())

fig, ax = plt.subplots(figsize=(7, 3.4))
analysis_df["event_type"].value_counts().reindex(["view","addtocart","transaction"]).plot.bar(ax=ax)
ax.set(title="Observed event types in sampled visitors", xlabel="Event type", ylabel="Number of events")
plt.tight_layout()
plt.show()

## 9 · Association rules — co-interactions by inferred session
The Retailrocket dataset does not supply genuine physical basket IDs or original product names. To avoid fabricating purchases, we mine **co-observed categories within visitor sessions**. We use confidence, support, and lift. A rule with lift > 1 indicates positive co-occurrence relative to an independence baseline; it is not evidence of causation. The analysis selects up to 40 popular known categories and uses sessions with at least two unique categories. Rules are reported only if observed; sparse datasets can legitimately produce none.

In [ ]:
category_pairs = pd.read_sql_query("""
SELECT DISTINCT e.session_id, i.category_id
FROM events e JOIN items i ON e.item_id=i.item_id
WHERE i.category_id IS NOT NULL
""", con)
category_frequency = category_pairs.category_id.value_counts()
top_categories = set(category_frequency.head(40).index.astype(int))
category_pairs = category_pairs[category_pairs.category_id.isin(top_categories)]
baskets = (category_pairs.groupby("session_id")["category_id"]
           .apply(lambda s: set(int(v) for v in s)).tolist())
baskets = [basket for basket in baskets if len(basket) >= 2]
print("Eligible inferred sessions for category associations:", len(baskets))
print("Each basket represents category co-interactions, NOT a confirmed purchase.")

if baskets:
    one_counts = Counter()
    pair_counts = Counter()
    for basket in baskets:
        one_counts.update(basket)
        pair_counts.update(combinations(sorted(basket),2))
    minimum_count = max(3, int(np.ceil(0.005 * len(baskets))))
    rules = []
    for (a, b), ab in pair_counts.items():
        if ab < minimum_count: continue
        for antecedent, consequent in ((a,b),(b,a)):
            support = ab / len(baskets)
            confidence = ab / one_counts[antecedent]
            lift = confidence / (one_counts[consequent] / len(baskets))
            rules.append({"antecedent_category": antecedent,
                          "consequent_category": consequent,
                          "support": support, "confidence": confidence,
                          "lift": lift, "matching_sessions": ab})
    rules_df = pd.DataFrame(rules)
    if not rules_df.empty:
        rules_df = rules_df.sort_values(["lift","confidence","support"],
                                         ascending=False).reset_index(drop=True)
        display(rules_df.head(15).round(4))
        rules_df.to_csv(OUTPUT_DIR / "session_category_association_rules.csv", index=False)
    else:
        print("No category pairs passed the minimum threshold; this is a valid negative result.")
else:
    rules_df = pd.DataFrame()
    print("Not enough distinct known categories across sessions for meaningful real-data rules.")

### Optional: Apriori / FP-Growth with `mlxtend`
For the standard textbook mining workflow, install using `%pip install mlxtend` in a separate notebook cell. This optional cell only runs when `mlxtend` is available, and always works from the **actual** category baskets defined above.

In [ ]:
try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import fpgrowth, association_rules
    if len(baskets) >= 2:
        te = TransactionEncoder()
        encoded = te.fit(baskets).transform(baskets)
        basket_matrix = pd.DataFrame(encoded, columns=te.columns_)
        frequent = fpgrowth(basket_matrix, min_support=max(0.005, 3/len(baskets)),
                             use_colnames=True, max_len=3)
        if not frequent.empty and (frequent.itemsets.map(len) >= 2).any():
            # Version-tolerant for older and newer mlxtend.
            try:
                assoc = association_rules(frequent, metric="confidence", min_threshold=0.01,
                                          num_itemsets=len(basket_matrix))
            except TypeError:
                assoc = association_rules(frequent, metric="confidence", min_threshold=0.01)
            display(assoc.sort_values("lift", ascending=False).head(10))
        else:
            print("No frequent multi-category itemsets at the chosen support.")
    else:
        print("Insufficient eligible baskets for mlxtend.")
except ImportError:
    print("Optional mlxtend is not installed. The dependency-free association calculations above already ran.")

## 10 · NoSQL document preparation and optional genuine MongoDB retrieval
The relational `events`, `visitors`, and `items` records can also be reshaped as nested JSON documents. JSON by itself is a file format, **not a NoSQL database**. The optional PyMongo cell below actually inserts and retrieves documents from a running MongoDB server. It is intentionally opt-in: the SQLite assignment works without MongoDB.

In [ ]:
# Export 2,000 nested documents in JSON Lines format (portable MongoDB import format).
# Preserve genuine fields; no sensitive usernames/emails are present in this source.
export_records = pd.read_sql_query("""
SELECT e.event_id,e.event_type,e.event_utc,e.transaction_id,e.session_id,
       e.intent_level,e.prior_session_events,e.visitor_id,v.total_events,
       e.item_id,i.category_id
FROM events e JOIN visitors v ON v.visitor_id=e.visitor_id
JOIN items i ON i.item_id=e.item_id
ORDER BY e.event_ts_ms,e.event_id LIMIT 2000""", con)

mongo_docs = []
for r in export_records.itertuples(index=False):
    mongo_docs.append({
        "event_id": int(r.event_id), "event_type": r.event_type, "event_utc": r.event_utc,
        "transaction_id": None if pd.isna(r.transaction_id) else int(r.transaction_id),
        "session_id": r.session_id, "intent_level": r.intent_level,
        "prior_session_events": int(r.prior_session_events),
        "visitor": {"visitor_id": int(r.visitor_id), "observed_total_events": int(r.total_events)},
        "item": {"item_id": int(r.item_id),
                 "category_id": None if pd.isna(r.category_id) else int(r.category_id)}
    })
jsonl_path = OUTPUT_DIR / "events_nosql_documents.jsonl"
with jsonl_path.open("w", encoding="utf-8") as out:
    for doc in mongo_docs:
        out.write(json.dumps(doc, ensure_ascii=False) + "\n")
print(f"Exported {len(mongo_docs)} document records to:", jsonl_path)
print("Example document:")
print(json.dumps(mongo_docs[0], indent=2))

In [ ]:
# OPTIONAL REAL NoSQL DB: To use, first run `%pip install pymongo`,
# start MongoDB locally (or provide a secure, appropriately scoped URI),
# and set RUN_MONGODB = True. Otherwise this step is skipped safely.
RUN_MONGODB = False
if RUN_MONGODB:
    from pymongo import MongoClient
    client = MongoClient("mongodb://localhost:27017/", serverSelectionTimeoutMS=3000)
    client.admin.command("ping")  # Verify live MongoDB, not a JSON-only demonstration.
    collection = client["retailrocket_project"]["events_demo"]
    collection.delete_many({})  # CAUTION: deletes only this deliberately named demo collection.
    collection.insert_many(mongo_docs)
    print("MongoDB documents:", collection.count_documents({}))
    print("MongoDB transaction example:", collection.find_one({"event_type": "transaction"}, {"_id": 0}))
    print("MongoDB nested query:", list(collection.find(
        {"visitor.observed_total_events": {"$gte": 2}},
        {"_id": 0, "event_id": 1, "visitor": 1}).limit(3)))
    client.close()
else:
    print("MongoDB demonstration is opt-in. Set RUN_MONGODB=True with pymongo and MongoDB installed.")

## 11 · Final deliverables and reflection

**Generated by this notebook** inside `retailrocket_outputs/`:
1. `retailrocket_project.sqlite` — a real SQLite database with normalized tables, foreign and composite keys.
2. `events_analysis_ready.csv` — query-derived, cleaned analytical extract.
3. `events_nosql_documents.jsonl` — nested NoSQL-ready JSON records.
4. `session_category_association_rules.csv` — generated only when real observed category pairs meet the defined frequency threshold.

**Evaluation notes:** Dataset sampling is reproducible but only representative of sampled visitor IDs; do not infer population-wide rates without appropriate weighting and validation. Session boundaries are inferred, not sourced. Heterogeneous item properties have temporal validity. Marketing data are fictional simulations. Category association is correlational, not evidence that one item causes another to be purchased.

**Sources and documentation:** [Retailrocket dataset](https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset) · [Python sqlite3](https://docs.python.org/3/library/sqlite3.html) · [SQLite FK documentation](https://www.sqlite.org/foreignkeys.html) · [mlxtend association rules](https://rasbt.github.io/mlxtend/user_guide/frequent_patterns/association_rules/).

In [ ]:
print("Finished successfully. Deliverables:")
for p in sorted(OUTPUT_DIR.iterdir()):
    if p.is_file(): print(f"{p.name:42s} {p.stat().st_size/1_000_000:9.2f} MB")
print("SQLite events:", con.execute("SELECT COUNT(*) FROM events").fetchone()[0])
con.commit()
con.close()